# Scratch test cleanup

Find test directories, review a selection, and permanently delete the selected
directories and all nested contents. The default layout is
`$PSCRATCH/noise2inverse360/<scan>/<test>`, including incomplete runs without
a `run_plan.json`. Discovery lists directories at a chosen depth; it does not
require test names to contain the word `test`.

Running all cells opens the browser and lists tests. Deletion requires a
reviewed selection, confirmation that associated jobs have stopped, and the
exact phrase `DELETE N`, where N is the number of selected directories.
There is no undo or trash folder.

## Setup

Requirements: Python 3.8 or newer, IPython, ipywidgets, and the repository's
`krg-ct-workflow` package. No Torch, image loading, or GPU is used.
In an existing NERSC ALS environment, run these terminal commands after
updating the checkout:

```bash
cd "$HOME/microct-n2i-source/KRG-alsDenoise"
"$HOME/microct-n2i-envs/als832/bin/python" -m pip install --no-deps -e workflow
```

The [NERSC setup guide](../docs/SETUP_NERSC.md) includes environment creation
and kernel registration for a new installation. Open this notebook with the
**ALS CT + N2I notebook** kernel and restart it after updating the helpers.
A separate kernel can browse scratch while training continues elsewhere,
but deletion requires stopping every job that uses the selected tests.
The cleanup browser checks filesystem locks; it does not query Slurm or
detect processes on other nodes.

In [ ]:
# 1. Import the scratch cleanup browser
import os
from pathlib import Path
import krg_workflow
from krg_workflow.scratch_cleanup import cleanup_browser

print('Workflow:', krg_workflow.__version__, '| source:', krg_workflow.__file__)


## Scratch directory and layout

Set `SCRATCH_ROOT` to the directory containing scan folders. With depth 2,
each `<scan>/<test>` directory is a separate selectable entry. To manage
tests for one scan, set the root to that scan's directory and use depth 1.
Depth 3 is available for an additional grouping level. An incorrect depth
can select a whole scan or an output subdirectory; the preview shows the
complete paths before deletion.

The root can also be entered as an absolute `Path`. `PSCRATCH` is read from
the NERSC environment; on another system, replace the expression below
with the scratch output path.

In [ ]:
# 2. Select the scratch root and directory depth
SCRATCH_ROOT = Path(os.environ['PSCRATCH']) / 'noise2inverse360'
TEST_DEPTH = 2


## Select, preview and delete

1. Click **Find tests** after changing the root, layout or name pattern.
   `*` includes every visible directory at the selected depth; `*test*`
   limits the list to matching names. Hidden directories are omitted.
2. Select entries with Ctrl/Cmd or Shift, then click **Preview deletion**.
   The preview shows full paths, file counts, logical sizes and the first
   20 top-level contents. Only selected trees are traversed to calculate size.
3. Stop jobs using those directories, check the confirmation box, and type
   the displayed `DELETE N` phrase. Click **Delete selected tests** once.
4. Review the completion report and click **Find tests** to refresh the list.

Symlinks, junctions, nested mount points, special files and run locks block
deletion. Changes to files or directories since preview require another
preview. Reconstruction workflow locks are held during cleanup. A stale
lock must be resolved separately after verifying that its owner stopped.

The scratch root and parent scan directories remain. Audit reports are saved
under `<scratch root>/.cleanup_reports/`. Queue entries are not rewritten;
remove entries for deleted runs with the HPC notebook's queue controls.
A filesystem error can leave a partially deleted test; the audit report lists
failed paths. Logical size can differ from space reclaimed on the filesystem.

In [ ]:
# 3. Open the test selection and deletion controls
cleanup_panel = cleanup_browser(SCRATCH_ROOT, depth=TEST_DEPTH)
